In [3]:
import re
import os
import paragraph_cleaning_tools as pct

In [4]:
os.makedirs('../clean_output', exist_ok=True)
list_text = []
stat_outputfile = os.path.join('../clean_output', 'stat_file')
with open(stat_outputfile, 'w', encoding='utf-8') as stat_file:
        stat_file.write('')
for i,file in enumerate(os.listdir('../test_output')) :
    if not file.startswith('good'):
            continue
    try:
        with open('../test_output'+'/'+file, 'r', encoding='utf-8') as infile:
            text = infile.read()
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    paragraphs = pct.paragraph_maker(text,maxpadding=1)
    paragraphs = pct.paragraph_clean_image(paragraphs)
    paragraphs = pct.paragraph_clean_dotlines(paragraphs)
    paragraphs = pct.paragraph_remove_artifacts(paragraphs)
    paragraphs = pct.paragraph_fix_broken_line(paragraphs)
    paragraphs = pct.paragraph_merger(paragraphs,500,10)
    paragraphs = pct.remove_numbered_title(paragraphs,pct.remove_title_number_pattern)
    paragraphs = pct.remove_taged_paragraphs(paragraphs=paragraphs,tags=pct.summary_tags,print=True)
    paragraphs = pct.remove_taged_paragraphs(paragraphs=paragraphs,tags=pct.catalog_tags,ending_tags=pct.CONTENT_and_CATALOG_end_tags,print=True)
    paragraphs = pct.remove_taged_paragraphs(paragraphs=paragraphs,tags=pct.bibliography_tags,print=True)
    paragraphs = pct.remove_taged_paragraphs(paragraphs=paragraphs,tags=pct.euritirio_tags,print=True)
    paragraphs = pct.remove_taged_paragraphs(paragraphs=paragraphs,tags=pct.glossary_tags,print=True)
    paragraphs = pct.remove_taged_paragraphs(paragraphs=paragraphs,tags=pct.content_tags,ending_tags=pct.CONTENT_and_CATALOG_end_tags,print=True,skip_paragraphs=1)
    paragraphs = pct.all_paragraph_not_char_end(paragraphs,pct.endings,print=True)
    paragraphs = pct.remove_noise(paragraphs,pct.noise_pattern)
    paragraphs = pct.paragraph_remove_artifacts(paragraphs)
    
    t_file = file

    output_file_path = os.path.join('../clean_output', t_file)

    try:
        with open(output_file_path, 'w', encoding='utf-8') as output_file:
            pct.test_write_text(paragraphs,output_file)
            list_text.append(pct.total_paragraphs(paragraphs))
        with open(stat_outputfile, 'a+', encoding='utf-8') as stat_file:
            pct.stat_assembly(pct.total_paragraphs(paragraphs),paragraphs)
            stat_file.write(t_file+' : '+str(pct.file_stat_list)+'\n')
        pct.file_stat_list = pct.file_reset_list()
    except Exception as e:
            print(f"Error writing to {output_file_path}: {e}")
            continue